# RFModelQ2

This notebook trains and evaluates the baseline Random Forest model for three-class Yelp sentiment classification.

In [2]:
import joblib
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

PART_B_DIR = Path.cwd().parents[0]
DATA_FILE = PART_B_DIR / "data" / "yelp_clean.csv"   # cleaned Yelp reviews from Q1
MODEL_DIR = PART_B_DIR / "models"

## Data Preparation

The cleaned reviews are loaded and divided into reproducible stratified training and test sets.

In [3]:
# --- Load the cleaned Yelp reviews and drop rows with no usable text ---
df = pd.read_csv(DATA_FILE, usecols=["clean_text", "sentiment"])

print(f"Reviews: {len(df)}")
print(df["sentiment"].value_counts().to_string())

Reviews: 29998
sentiment
positive    12000
negative    11998
neutral      6000


In [4]:
# --- Stratified 80:20 split. The same random_state is used in Q3 and Q4 so every
#     number reported in Part B refers to the identical held-out test set. ---
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["sentiment"],
    test_size=0.2, random_state=42, stratify=df["sentiment"]
)
print(f"Train: {len(X_train)}   Test: {len(X_test)}")

Train: 23998   Test: 6000


## Baseline Model Training

A TF-IDF and Random Forest pipeline is trained with its baseline configuration and saved for later comparison.

In [5]:
# --- Baseline pipeline built entirely from scikit-learn defaults, left untuned on purpose ---
# tfidf : no vocabulary cap, unigrams only, raw (non-sublinear) term frequency
# clf   : 100 trees, unlimited depth, sqrt(n_features) candidates per split, no class weighting
# This is the reference point that the Q3 hyperparameter search has to beat.
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", RandomForestClassifier(random_state=42, n_jobs=-1)),
])

# --- Fit, save for the Q3/Q4 comparison, then predict on the held-out test set ---
pipeline.fit(X_train, y_train)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline, MODEL_DIR / "rf_pipeline.joblib")
preds = pipeline.predict(X_test)

print(f"TF-IDF vocabulary size: {len(pipeline.named_steps['tfidf'].vocabulary_)}")

TF-IDF vocabulary size: 38031


## Model Evaluation

The baseline model is evaluated on the unseen test set using per-class and overall classification metrics.

In [6]:
# --- Per-class performance ---
print("=== Q2: Baseline Yelp Random Forest 3-Class Classification Report ===")
print(classification_report(y_test, preds, digits=4, zero_division=0))

# --- Headline measures. Macro averages matter most here because neutral has only half
#     as many reviews as negative and positive, so accuracy alone would flatter the model.
macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
    y_test, preds, average="macro", zero_division=0
)
weighted_f1 = precision_recall_fscore_support(
    y_test, preds, average="weighted", zero_division=0
)[2]

print("=== Q2: Baseline Yelp Random Forest Summary ===")
print(f"Accuracy:        {accuracy_score(y_test, preds):.4f}")
print(f"Macro Precision: {macro_p:.4f}")
print(f"Macro Recall:    {macro_r:.4f}")
print(f"Macro F1-score:  {macro_f1:.4f}")
print(f"Weighted F1:     {weighted_f1:.4f}")

=== Q2: Baseline Yelp Random Forest 3-Class Classification Report ===
              precision    recall  f1-score   support

    negative     0.6892    0.8954    0.7789      2400
     neutral     0.6364    0.0350    0.0664      1200
    positive     0.7301    0.8567    0.7883      2400

    accuracy                         0.7078      6000
   macro avg     0.6852    0.5957    0.5445      6000
weighted avg     0.6950    0.7078    0.6402      6000

=== Q2: Baseline Yelp Random Forest Summary ===
Accuracy:        0.7078
Macro Precision: 0.6852
Macro Recall:    0.5957
Macro F1-score:  0.5445
Weighted F1:     0.6402
